# 06 - Régression linéaire

On estime l’effet du contexte social et du retard scolaire sur le score de lecture.
La formule de base est :
score_lecture ~ pcs + sexe + retard


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.loader import load_csv, data_path

df = load_csv(data_path('data', 'interim', 'etude_lecture_6e_clean.csv'))

df['sexe'] = df['sexe'].map({'F': 0, 'M': 1})
df['retard'] = df['retard'].astype(int)

dummy_pcs = pd.get_dummies(df['pcs'], prefix='pcs', drop_first=True).astype(float)
model_df = pd.concat([df[['score_lecture', 'sexe', 'retard']], dummy_pcs], axis=1)
X = model_df.drop(columns=['score_lecture'])
y = model_df['score_lecture']
X = X.copy()
X.insert(0, 'const', 1.0)
# Ensure exogenous variables are numeric (coerce any unexpected object dtype)
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
# Ensure target is numeric and align indices
y = pd.to_numeric(y, errors='coerce')
# Drop rows with NaNs just in case
valid_idx = X.index[~(X.isnull().any(axis=1) | y.isnull())]
X = X.loc[valid_idx]
y = y.loc[valid_idx]

coefficients, *_ = np.linalg.lstsq(X.to_numpy(), y.to_numpy(), rcond=None)
print('Coefficients OLS :')
print(pd.Series(coefficients, index=X.columns).round(4))


Coefficients OLS :
const                             69.2686
sexe                              -1.5869
retard                           -19.9805
pcs_Cadre                         15.9272
pcs_Employe                       -0.6716
pcs_Ouvrier                       -7.8466
pcs_Professions_intermediaires     6.1821
pcs_Retraite                      -3.7428
dtype: float64
